install dependencies

In [1]:
%pip install -q python-terrier ir_datasets_longeval pyterrier-dr

imports & initialization

In [2]:
import pyterrier as pt
import pyterrier_dr as pt_dr
from ir_datasets_longeval import load
import pandas as pd

load dataset and create dense embedding index

In [4]:
if not pt.started():
    pt.init()

dataset = load("longeval-sci-2026/snapshot-1/train/dctr")

# Maintain a reference to the ir_datasets store for retrieving text later
docs_store = dataset.docs_store()

# Initialize the embedding model (TAS-B dual-encoder)
model = pt_dr.TasB()
# Create a dense indexer mapping embeddings to local disk
indexer = pt_dr.NumpyIndex("./dense_index")

# The pipeline: pass documents through the model's document encoder, then index
index_pipeline = model.doc_encoder() >> indexer

# Limit indexing to the first N documents. Set to None for no limit.
LIMIT_DOCS = 10000

# If True, only index documents that are judged in qrels. This allows fast and accurate IR evaluations on the query sets.
FILTER_TO_QRELS = False

def document_generator():
    judged_docnos = set()
    if FILTER_TO_QRELS:
        try:
            qrels_df = pd.DataFrame(dataset.qrels)
            # Handle potential variation in column naming (doc_id vs docno)
            doc_col = "doc_id" if "doc_id" in qrels_df.columns else "docno"
            judged_docnos = set(qrels_df[doc_col].unique())
            print(f"Filtering corpus: indexing only {len(judged_docnos)} judged documents...")
        except Exception as e:
            print(f"Warning: Could not load qrels for filtering: {e}. Indexing all docs.")
            
    count = 0
    for doc in dataset.docs_iter():
        # Apply qrels filter if enabled
        if FILTER_TO_QRELS and doc.doc_id not in judged_docnos:
            continue
            
        yield {
            "docno": doc.doc_id,
            "text": doc.default_text()
        }
        
        count += 1
        # Apply sample limit if enabled
        if LIMIT_DOCS is not None and count >= LIMIT_DOCS:
            print(f"Reached LIMIT_DOCS of {LIMIT_DOCS}. Stopping indexing.")
            break

# Execute indexing (Note: Dense indexing is computationally heavy; CPU execution will be slow)
index_pipeline.index(document_generator())

/tmp/ipykernel_10319/1195323437.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Reached LIMIT_DOCS of 10000. Stopping indexing.


create the search system pipeline

In [5]:
# The retrieval pipeline: encode the string query, then search against the dense index
dense_retriever = model.query_encoder() >> indexer

search the system and log the output

In [8]:
query = input("Enter search query: ").strip()

if not query:
    print("Empty query - skipping search.")
else:
    # ── Run dense retrieval ────────────────────────────────────
    results = dense_retriever.search(query)

    print("\n", "=" * 70)
    print(f"  QUERY : {query!r}")
    print(f"  Results returned : {len(results)}")
    print("=" * 70)

    # ── Top-10 results ──────────────────────────────────────
    TOP_N       = 10
    SNIPPET_LEN = 300   # characters of 'text' to show

    top = results.head(TOP_N).reset_index(drop=True)

    for _, row in top.iterrows():
        rank  = int(row["rank"])
        docno = row["docno"]
        score = row["score"]

        # Retrieve stored text natively via ir_datasets instead of PyTerrier meta-index
        try:
            doc_obj = docs_store.get(docno)
            text = doc_obj.default_text() if doc_obj else "<not found in docs_store>"
        except Exception as e:
            text = f"<text unavailable: {e}>"

        snippet = (text[:SNIPPET_LEN] + "…") if text and len(text) > SNIPPET_LEN else text

        print(f"  [{rank:>2}]  docno={docno}  score={score:.6f}")
        print(f"        {snippet}")
        print()



  QUERY : 'trump'
  Results returned : 1000
  [ 0]  docno=289579227  score=100.180878
        Analyzing the Role of Donald Trump as a Catalyst for the Surging European Far Right Only about a quarter of the world population is currently living under a democracy, down from 54 percent in 2006 (Owens, 2023). In the United States, President Donald Trump’s sweeping executive actions and increasing…

  [ 1]  docno=289602786  score=97.878265
        Tourism resilience in Iran:navigating sanctions, diplomacy and emerging opportunities The re-election of Donald Trump as the 47th president of the United States in 2024 marks a significant shift in US-Iran relations, with far-reaching consequences for Iran’s tourism sector. This brief report examine…

  [ 2]  docno=289545175  score=96.366631
        Looking ahead:imbalance, dependency and NATO’s uncertain future Throughout is 75-year history, NATO’s European and Canadian allies have had a dependent and uneasy relationship with the United States. T